In [1]:
from os.path import realpath

# Local steps to run in pycharm
# conda activate sage
# jupyter lab --no-browser --port=8888
# configure external server with the token
# sagemath (10.7)

from sage.all import *
from sage.plot.plot import graphics_array
from itertools import combinations
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def graph_square(H):
    G = Graph()
    G.add_vertices(H.vertices())
    for u in H.vertices():
        for v in H.vertices():
            if u < v and H.distance(u,v) <= 2:
                G.add_edge(u,v)
    return G

def is_square_root(G, H):
    return graph_square(H) == G

In [2]:
# Efficient find_square_root function

def violates_partial(G, H):
    for u, v in combinations(H.vertices(), 2):
        try:
            d = H.distance(u, v)
        except (ValueError, RuntimeError):
            continue

        if H.has_edge(u, v) and not G.has_edge(u, v):
            return True
        if d <= 2 and not G.has_edge(u, v):
            return True
    return False

def find_square_root(G):
    V = list(G.vertices())
    pairs = list(combinations(V, 2))
    H = Graph()
    H.add_vertices(V)

    def backtrack(idx):
        if idx == len(pairs):
            return H.copy() if is_square_root(G, H) else None

        u, v = pairs[idx]

        H.add_edge(u, v)
        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result
        H.delete_edge(u, v)

        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result

        return None

    return backtrack(0)

In [3]:
# LP with max objective function
def max_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

# LP with max objective function and additional triangle constraints for line graph root structure
def max_triangular_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    # Count triangles and consider vertices in only one triangle O(n^3)
    deg = {v: G.degree(v) for v in range(n)}
    triangles_at = {v: [] for v in range(n)}

    for i in range(n):
        for j in range(i+1, n):
            if not G.has_edge(i, j):
                continue
            for k in range(j+1, n):
                if G.has_edge(i, k) and G.has_edge(j, k):
                    triangles_at[i].append((j, k))
                    triangles_at[j].append((i, k))
                    triangles_at[k].append((i, j))

    for s, tris in triangles_at.items():
        if len(tris) == 1:
            u, v = tris[0]
            if deg[u] < deg[v]:
                w = u
            elif deg[v] < deg[u]:
                w = v
            else:
                continue
            m.add_constraint(x[s,w] == 1)
            m.add_constraint(x[w,s] == 1)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

In [4]:
# Generate all possible graph squares/roots and their complements for given n
import os
import matplotlib.pyplot as plt
import itertools
from matplotlib.backends.backend_pdf import PdfPages
from sage.graphs.graph_generators import graphs as graph_gens
from sage.numerical.mip import MIPSolverException

n = 6
ncpus = 12
dir = f"max_isomorphism_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment='center')
    g.axes(False)
    return g

def fmt(x, width=8, digits=3):
    if x is None:
        return f"{'NA':>{width}}"
    return f"{x:>{width}.{digits}f}"

def process_graph(idx, G, label):
    G.relabel(range(G.order()))
    n = G.order()

    A_H = None
    sol_H = None
    H = None
    lp_status = "unknown"
    has_root = False
    edges_H = None
    is_noninteger = False

    try:
        m, x = max_linear_program(G)
        m.solve()
        solution_x = m.get_values(x)
        A_H = matrix([[solution_x[i, j] for j in range(n)] for i in range(n)])
        sol_H = Graph(A_H, weighted=True)
        for (u, v, w) in sol_H.edges():
            sol_H.set_edge_label(u, v, round(w, 3))

        p1 = sol_H.plot(edge_labels=True, title='Weighted LP H')
        lp_status = "feasible"

        if any(abs(A_H[i, j] - round(A_H[i, j])) > 1e-6 for i in range(n) for j in range(n)):
            is_noninteger = True

    except MIPSolverException:
        sol_H = None
        p1 = error_plot("LP infeasible")
        lp_status = "infeasible"

    H = find_square_root(G)

    if H is None:
        p2 = error_plot("No feasible H")
        has_root = False
        edges_H = None
    else:
        p2 = H.plot(title='Feasible H')
        has_root = True
        edges_H = H.size()

    p3 = G.plot(title=f"{label} G (#{idx})")

    summary = {
        "index": idx,
        "type": label,
        "edges_G": G.size(),
        "vertices_G": n,
        "lp_status": lp_status,
        "has_root": has_root,
        "edges_H": edges_H,
        "is_noninteger": is_noninteger
    }

    is_mismatch = (is_noninteger or (lp_status == "feasible" and H is None))

    if is_mismatch:
        if sol_H is not None:
            p_left = sol_H.plot(edge_labels=True, title="Weighted LP H")
        else:
            p_left = error_plot("No LP solution")

        if H is not None:
            p_mid = H.plot(title="Feasible H")
        else:
            p_mid = error_plot("No discrete H")

        p_right = G.plot(title="Original G")
        comparison = graphics_array([p_left, p_mid, p_right])
    else:
        comparison = None

    return (p1, p2, p3), summary, comparison, is_mismatch

def analyze_isomorphic_lp(G):
    n = G.order()
    perms = list(itertools.permutations(range(n)))

    results = []
    integer_count = 0
    fractional_count = 0
    infeasible_count = 0

    for perm in perms:

        Gp = G.relabel(dict(zip(range(n), perm)), inplace=False)

        try:
            m, x = max_linear_program(Gp)
            m.solve()
            sol = m.get_values(x)

            A = matrix([[sol[i, j] for j in range(n)] for i in range(n)])

            is_fractional = any(
                abs(A[i, j] - round(A[i, j])) > 1e-6
                for i in range(n)
                for j in range(n)
            )

            if is_fractional:
                fractional_count += 1
                results.append(("fractional", perm))
            else:
                integer_count += 1
                results.append(("integer", perm))

        except MIPSolverException:
            infeasible_count += 1
            results.append(("infeasible", perm))

    diagnostics = {
        "total_perms": len(perms),
        "integer": integer_count,
        "fractional": fractional_count,
        "infeasible": infeasible_count,
        "details": results
    }

    return diagnostics

def run_task(idx, G, label):
    return process_graph(idx, G, label)

tasks = [(i, Graph(G0)) for i, G0 in enumerate(graph_gens.nauty_geng(str(n)))]

results = []
mismatch_results = []

for idx, G in tasks:
    plot_G, summary_G, comp_G, mismatch_G = run_task(idx, G, label="Original")
    if (mismatch_G and comp_G is not None) and summary_G["has_root"]:
        iso_diag = analyze_isomorphic_lp(G)
        summary_G["iso_analysis"] = iso_diag
        mismatch_results.append((comp_G, summary_G))

    (p1, p2, p3) = plot_G

    combined = graphics_array([[p1, p2, p3]])
    results.append((combined, summary_G))

    print(
        f"✅ Graph {idx:>4} | "
        f"LP={summary_G['lp_status']} | "
        f"Root={summary_G['has_root']} | "
        f"{'⚠️ mismatch' if mismatch_G else ''}"
    )


with open(summary_file, "w") as f:

    header = (
        f"{'Graph':>5} | {'Verts':>5} | "
        f"{'Edges(G)':>8} | {'Edges(H)':>8} | "
        f"{'LP':>10} | {'Root':>5} | {'NonInt':>7}\n"
    )

    f.write(header)
    f.write("-" * len(header) + "\n")

    for _, sG in results:

        f.write(
            f"{sG['index']:>5} | "
            f"{sG['vertices_G']:>5} | "
            f"{sG['edges_G']:>8} | "
            f"{str(sG['edges_H']) if sG['edges_H'] is not None else 'NA':>8} | "
            f"{sG['lp_status']:<10} | "
            f"{'Yes' if sG['has_root'] else 'No':>5} | "
            f"{'Yes' if sG['is_noninteger'] else 'No':>7}\n"
        )

        if "iso_analysis" in sG:

            iso = sG["iso_analysis"]

            f.write(
                f"      ISO CHECK → perms={iso['total_perms']} | "
                f"integer={iso['integer']} | "
                f"fractional={iso['fractional']} | "
                f"infeasible={iso['infeasible']}\n"
            )

        f.write("-" * len(header) + "\n")

print(f"📝 Summary written to {summary_file}")

with PdfPages(output_pdf) as pdf:
    for p, sG in results:
        fig = p.matplotlib(figsize=(12, 4))
        pdf.savefig(fig)
        plt.close(fig)
print(f"\n✅ Combined {len(results)} graphs into {output_pdf}")

if mismatch_results:
    with PdfPages(comparison_pdf) as pdf:
        for comp, s in mismatch_results:

            fig = comp.matplotlib(figsize=(12, 4))

            plt.suptitle(
                f"Graph {s['index']} — "
                f"LP={s['lp_status']} | "
                f"Root={s['has_root']} | "
                f"{'Non-integer LP' if s['is_noninteger'] else ''}"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"⚠️ Comparison mismatches saved to {comparison_pdf}")

✅ Graph    0 | LP=feasible | Root=True | 
✅ Graph    1 | LP=feasible | Root=True | 
✅ Graph    2 | LP=infeasible | Root=False | 
✅ Graph    3 | LP=infeasible | Root=False | 
✅ Graph    4 | LP=infeasible | Root=False | 
✅ Graph    5 | LP=infeasible | Root=False | 
✅ Graph    6 | LP=feasible | Root=True | 
✅ Graph    7 | LP=infeasible | Root=False | 
✅ Graph    8 | LP=infeasible | Root=False | 
✅ Graph    9 | LP=feasible | Root=True | 
✅ Graph   10 | LP=infeasible | Root=False | 
✅ Graph   11 | LP=infeasible | Root=False | 
✅ Graph   12 | LP=infeasible | Root=False | 
✅ Graph   13 | LP=infeasible | Root=False | 
✅ Graph   14 | LP=infeasible | Root=False | 
✅ Graph   15 | LP=infeasible | Root=False | 
✅ Graph   16 | LP=infeasible | Root=False | 
✅ Graph   17 | LP=infeasible | Root=False | 
✅ Graph   18 | LP=infeasible | Root=False | 
✅ Graph   19 | LP=infeasible | Root=False | 
✅ Graph   20 | LP=feasible | Root=True | 
✅ Graph   21 | LP=infeasible | Root=False | 
✅ Graph   22 | LP=infeasi

<Figure size 640x480 with 0 Axes>

In [5]:
# Analyze graph summary with isomorphism
from collections import defaultdict

filename = "max_isomorphism_graph_output/n=6/graph_summary_n=6.txt"

summary = defaultdict(list)
iso_effects = []
original_status = {}

current_graph = None

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith("-") or line.startswith("SUMMARY") or line.startswith("Graph"):
            continue

        if line.startswith("ISO CHECK"):
            if current_graph is None:
                continue

            parts = [p.strip() for p in line.replace("ISO CHECK →", "").split("|")]

            iso_data = {}
            for p in parts:
                key, value = p.split("=")
                iso_data[key.strip()] = int(value.strip())

            iso_effects.append((current_graph, iso_data))
            continue

        parts = [p.strip() for p in line.split("|")]
        if len(parts) != 7:
            continue

        graph, verts, edges_g, edges_h, lp, root, nonint = parts
        current_graph = graph

        H_exists = (root == "Yes")

        if lp == "infeasible":
            LP_type = "LP infeasible"
        elif nonint == "Yes":
            LP_type = "LP fractional"
        else:
            LP_type = "LP integer"

        original_status[graph] = (H_exists, LP_type)
        summary[(H_exists, LP_type)].append(graph)

for (H_exists, LP_type), graphs in sorted(summary.items()):
    print(
        f"H exists: {str(H_exists):<5} | "
        f"{LP_type:<15} → Count: {len(graphs)}"
    )


if not iso_effects:
    print("No ISO checks found.")
else:
    for graph, iso_data in iso_effects:

        H_exists, LP_type = original_status.get(graph, ("Unknown", "Unknown"))

        if not (H_exists is True and LP_type == "LP fractional"):
            continue

        filtered = True

        print(f"\nGraph {graph}")
        print("  Original → H exists: True | LP fractional")

        print("  ISO permutations:", iso_data["perms"])
        print("     Integer:", iso_data["integer"])
        print("     Fractional:", iso_data["fractional"])
        print("     Infeasible:", iso_data["infeasible"])


H exists: False | LP fractional   → Count: 21
H exists: False | LP infeasible   → Count: 107
H exists: True  | LP fractional   → Count: 4
H exists: True  | LP integer      → Count: 24

Graph 115
  Original → H exists: True | LP fractional
  ISO permutations: 720
     Integer: 0
     Fractional: 720
     Infeasible: 0

Graph 130
  Original → H exists: True | LP fractional
  ISO permutations: 720
     Integer: 272
     Fractional: 448
     Infeasible: 0

Graph 139
  Original → H exists: True | LP fractional
  ISO permutations: 720
     Integer: 40
     Fractional: 680
     Infeasible: 0

Graph 153
  Original → H exists: True | LP fractional
  ISO permutations: 720
     Integer: 112
     Fractional: 608
     Infeasible: 0
